# Tests for Natural Earth datas downloading

In [ ]:
from typing import Any
import tomllib
import requests

with open("../src/zones.toml", "rb") as f:
    zones_config = tomllib.load(f)

base_path = zones_config["source_path"]

print(base_path)


In [ ]:
def build_next_url(config: dict[str, Any], family_name: str, name: str, precision_level: str = "medium") -> str:
    precision_mapping = {
        "low": 110,
        "medium": 50,
        "high": 10,
    }
    family = config["data_family"].get(family_name, [])
    theme = next(
        item for item in family
        if item["name"] == name
    )
    authorized_precision_levels = theme.get("precision", [])
    if precision_level not in authorized_precision_levels:
        precision_level = "medium"
    precision = precision_mapping[precision_level]
    return f'{precision}m_{family_name}/ne_{precision}m_{theme["link"]}.zip'

In [ ]:


session = requests.Session()
theme_name = "cultural"
next_url = build_next_url(zones_config, theme_name, "Countries")
print(f'{base_path}/{next_url}')


In [ ]:
from zipfile import ZipFile
import io
from shapefile import Reader

response = session.get(f'{base_path}/{next_url}', timeout=10)
response.raise_for_status()
archive = ZipFile(io.BytesIO(response.content))
base_name = next_url.rsplit("/", 1)[-1].split(".")[0]
for file_name in archive.namelist():
    if file_name.startswith(base_name) and file_name.split(".")[-1] in ["shp", "shx", "dbf"]:
        with archive.open(file_name, mode='r') as f:
            with open(file_name, "wb") as out_file:
                out_file.write(f.read())